# Ranking with XGBoost


- Load Amazon Beauty train/valid/test interaction splits plus item metadata
- Perform 1:1 negative sampling, categorical encoding, and timestamp normalization
- Train/evaluate an XGBoost ranker with automatic GPU/CPU selection
- Persist the trained booster and metrics under a configurable output directory




In [1]:
import json
import os
import random
import subprocess
from collections import defaultdict

import numpy as np
import pandas as pd
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.preprocessing import LabelEncoder

try:
    import xgboost as xgb
except ImportError as exc:
    raise ImportError(
        "Install xgboost (CPU or GPU build) before running."
    ) from exc

print("Using XGBoost version:", xgb.__version__)



Using XGBoost version: 3.1.2


In [2]:

# Configurable paths & hyperparameters 

DATA_DIR = os.environ.get("DATA_DIR", "dataset/amazon-beauty")
INTER_PREFIX = os.path.join(DATA_DIR, "amazon-beauty")
ITEM_FILE = os.environ.get("ITEM_FILE", f"{INTER_PREFIX}.item")
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", "saved_models")
NUM_NEG = int(os.environ.get("NUM_NEG", 1))
SEED = int(os.environ.get("SEED", 42))
VALID_SEED = SEED + 1
TEST_SEED = SEED + 2
GPU_ID = int(os.environ.get("GPU_ID", 0))

os.makedirs(OUTPUT_DIR, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)

print(f"DATA_DIR: {DATA_DIR}")
print(f"ITEM_FILE: {ITEM_FILE}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"NUM_NEG: {NUM_NEG}")
print(f"SEED: {SEED}")

# --- Build RecBole integer -> token mapping for both users and items ---
# RecBole converts token IDs to integers by sorting unique tokens alphabetically (0-based)
# We need this mapping to convert integer IDs back to original tokens for proper feature merging
_inter_orig = pd.read_csv(os.path.join(DATA_DIR, "amazon-beauty.inter"), sep="\t")

# Build item mapping
_unique_items = _inter_orig["item_id:token"].unique()
_sorted_items = sorted(_unique_items)
INT_TO_ITEM = {i: token for i, token in enumerate(_sorted_items)}
ITEM_TO_INT = {token: i for i, token in enumerate(_sorted_items)}
print(f"Built item ID mapping: {len(INT_TO_ITEM)} items")

# Build user mapping
_unique_users = _inter_orig["user_id:token"].unique()
_sorted_users = sorted(_unique_users)
INT_TO_USER = {i: token for i, token in enumerate(_sorted_users)}
USER_TO_INT = {token: i for i, token in enumerate(_sorted_users)}
print(f"Built user ID mapping: {len(INT_TO_USER)} users")

del _inter_orig, _unique_items, _sorted_items, _unique_users, _sorted_users  # Clean up temp variables



DATA_DIR: dataset/amazon-beauty
ITEM_FILE: dataset/amazon-beauty/amazon-beauty.item
OUTPUT_DIR: saved_models
NUM_NEG: 1
SEED: 42
Built item ID mapping: 249274 items
Built user ID mapping: 1210271 users


In [4]:
"""Read RecBole interaction split (.inter) and convert integer IDs to original tokens."""
def load_interactions(split: str) -> pd.DataFrame:
    
    path = f"{INTER_PREFIX}-{split}.inter"
    df = pd.read_csv(path, sep="\t")
    if "label" not in df.columns:
        raise ValueError(f"'label' column not found in {path}")
    
    # Convert integer item_ids to ASINs using the pre-built mapping
    df["item_id"] = df["item_id"].map(INT_TO_ITEM)
    unmapped_items = df["item_id"].isna().sum()
    if unmapped_items > 0:
        print(f"Warning: {unmapped_items} items could not be mapped in {split}")
    
    # Convert integer user_ids to original tokens
    df["user_id"] = df["user_id"].map(INT_TO_USER)
    unmapped_users = df["user_id"].isna().sum()
    if unmapped_users > 0:
        print(f"Warning: {unmapped_users} users could not be mapped in {split}")
    
    # Drop rows with unmapped IDs
    df = df.dropna(subset=["item_id", "user_id"])
    
    return df

"""Load item metadata and keep the fields used by the ranker."""
def load_item_features() -> pd.DataFrame:
    
    item_df = pd.read_csv(ITEM_FILE, sep="\t")
    rename_map = {
        "item_id:token": "item_id",
        "sales_rank:float": "sales_rank",
        "price:float": "price",
        "brand:token": "brand",
        "categories:token_seq": "categories",
    }
    item_df = item_df.rename(columns=rename_map)
    return item_df["item_id sales_rank price brand categories".split()]


def extract_primary_category(cat_str: str) -> str:
    if pd.isna(cat_str) or not str(cat_str).strip():
        return "Unknown"
    tokens = [c.strip().strip("'\"") for c in str(cat_str).split(",")]
    return tokens[0] if tokens else "Unknown"

"""Cache positive items per user across splits for neg sampling."""
def build_user_pos_items(df_list):
    
    user_pos = defaultdict(set)
    for df in df_list:
        for row in df[["user_id", "item_id"]].itertuples(index=False):
            user_pos[row.user_id].add(row.item_id)
    return user_pos

"""Uniform negative sampling per interaction (1 negative per positive)."""
def sample_negatives(df, user_pos_items, all_items, num_neg=1, seed=42):
    
    rng = random.Random(seed)
    negatives = []
    for row in df.itertuples(index=False):
        user = row.user_id
        for _ in range(num_neg):
            while True:
                neg_item = rng.choice(all_items)
                if neg_item not in user_pos_items[user]:
                    negatives.append(
                        {
                            "user_id": user,
                            "item_id": neg_item,
                            "timestamp": row.timestamp,
                            "label": 0,
                        }
                    )
                    break
    neg_df = pd.DataFrame(negatives)
    return pd.concat([df, neg_df], ignore_index=True)



In [5]:
def encode_features(train_df, valid_df, test_df, item_df):
    item_df = item_df.copy()
    item_df["primary_category"] = item_df["categories"].apply(extract_primary_category)
    item_df["price"] = pd.to_numeric(item_df["price"], errors="coerce")
    item_df["sales_rank"] = pd.to_numeric(item_df["sales_rank"], errors="coerce")
    item_df["price"] = item_df["price"].fillna(item_df["price"].median()) # why median: Price distributions are often right-skewed, mean is pulled up by outliers; median is not 
    item_df["sales_rank"] = item_df["sales_rank"].fillna(item_df["sales_rank"].median())
    item_df["brand"] = item_df["brand"].fillna("Unknown")
    item_df["primary_category"] = item_df["primary_category"].fillna("Unknown")

    def merge_item_feat(df):
        return df.merge(item_df, on="item_id", how="left")

    train_df = merge_item_feat(train_df)
    valid_df = merge_item_feat(valid_df)
    test_df = merge_item_feat(test_df)

    cat_cols = ["user_id", "item_id", "brand", "primary_category"]
    # Ensure every categorical column is string before concatenation/encoding
    for df in (train_df, valid_df, test_df):
        for col in cat_cols:
            df[col] = df[col].astype(str)

    encoders = {col: LabelEncoder() for col in cat_cols}
    combined = pd.concat([train_df[cat_cols], valid_df[cat_cols], test_df[cat_cols]])
    for col in cat_cols:
        # Step 1: FIT - Learn the mapping: This builds a vocabulary: {"user_123": 0, "user_456": 1, ...}
        encoders[col].fit(combined[col])
        # Step 2: TRANSFORM - Apply the mapping: This converts: "user_123" → 0, "user_456" → 1, ...
        train_df[col + "_idx"] = encoders[col].transform(train_df[col])
        valid_df[col + "_idx"] = encoders[col].transform(valid_df[col])
        test_df[col + "_idx"] = encoders[col].transform(test_df[col])

    for df in (train_df, valid_df, test_df):
        df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce").fillna(0)
        df["timestamp_days"] = (df["timestamp"] / 86400).astype(np.float32) #converts Unix timestamps (seconds since epoch) to days, Other features (e.g., price, sales_rank) are on different scales(price: 25.99 small scale, if use timestamps is large scale:1388534400) 

    feature_cols = [
        "user_id_idx",
        "item_id_idx",
        "price",
        "sales_rank",
        "brand_idx",
        "primary_category_idx",
        "timestamp_days",
    ]

    return train_df, valid_df, test_df, feature_cols

"""Auto-select GPU or CPU device (XGBoost 3.1+ API)."""
def detect_device():
    try:
        result = subprocess.run(
            ["nvidia-smi"], capture_output=True, text=True, timeout=2, check=False
        )
        gpu_present = result.returncode == 0
    except FileNotFoundError:
        gpu_present = False
    except Exception:
        gpu_present = False

    if gpu_present:
        device = f"cuda:{GPU_ID}"
        print(f"GPU detected - using device={device}")
        return device
    else:
        print("No GPU detected - using device=cpu")
        return "cpu"


def train_xgboost(train_df, valid_df, feature_cols):
    train_dmatrix = xgb.DMatrix(train_df[feature_cols], label=train_df["label"])
    valid_dmatrix = xgb.DMatrix(valid_df[feature_cols], label=valid_df["label"])

    params = {
        "objective": "binary:logistic", # binary classification with logistic regression output (probabilities 0 to 1 via sigmoid)
        "eval_metric": ["auc", "logloss"], # area under ROC curve (AUC) (higher is better, 0-1), LogLoss (lower is better)
        "device": detect_device(), 
        "tree_method": "hist", # histogram method (works for both CPU and GPU in 3.1+)
        "eta": 0.05, # learning rate
        "max_depth": 8, # Maximum depth of each tree
        "subsample": 0.8, # Fraction of training samples used per tree (Uses 80% of rows per tree)
        "colsample_bytree": 0.8, # Fraction of features used per tree
        "min_child_weight": 3, #Minimum sum of instance weights (Hessian) in a child node, for binary classification, roughly minimum samples per leaf, prevents splits that create very small leaves
        "lambda": 1.0, # L2 regularization on leaf weights, penalizes large leaf values, prevents overfitting
    }
    evals = [(train_dmatrix, "train"), (valid_dmatrix, "valid")]
    model = xgb.train(
        params,
        train_dmatrix,
        num_boost_round=500,
        evals=evals,
        early_stopping_rounds=30,
        verbose_eval=50, # Print metrics every 50 rounds
    )
    return model


def evaluate_model(model, df, feature_cols):
    dmatrix = xgb.DMatrix(df[feature_cols])
    preds = model.predict(dmatrix)
    # Clip predictions to avoid log(0) issues
    preds_clipped = np.clip(preds, 1e-15, 1 - 1e-15)
    auc = roc_auc_score(df["label"], preds)
    ll = log_loss(df["label"], preds_clipped)
    return {"AUC": float(auc), "LogLoss": float(ll)}



In [6]:
def encode_features_v2(train_df, valid_df, test_df, item_df):
    """Same as encode_features but returns encoders and item_lookup for saving."""
    item_df = item_df.copy()
    item_df["primary_category"] = item_df["categories"].apply(extract_primary_category)
    item_df["price"] = pd.to_numeric(item_df["price"], errors="coerce")
    item_df["sales_rank"] = pd.to_numeric(item_df["sales_rank"], errors="coerce")
    item_df["price"] = item_df["price"].fillna(item_df["price"].median())
    item_df["sales_rank"] = item_df["sales_rank"].fillna(item_df["sales_rank"].median())
    item_df["brand"] = item_df["brand"].fillna("Unknown")
    item_df["primary_category"] = item_df["primary_category"].fillna("Unknown")

    def merge_item_feat(df):
        return df.merge(item_df, on="item_id", how="left")

    train_df = merge_item_feat(train_df)
    valid_df = merge_item_feat(valid_df)
    test_df = merge_item_feat(test_df)

    cat_cols = ["user_id", "item_id", "brand", "primary_category"]
    for df in (train_df, valid_df, test_df):
        for col in cat_cols:
            df[col] = df[col].astype(str)

    encoders = {col: LabelEncoder() for col in cat_cols}
    combined = pd.concat([train_df[cat_cols], valid_df[cat_cols], test_df[cat_cols]])
    for col in cat_cols:
        encoders[col].fit(combined[col])
        train_df[col + "_idx"] = encoders[col].transform(train_df[col])
        valid_df[col + "_idx"] = encoders[col].transform(valid_df[col])
        test_df[col + "_idx"] = encoders[col].transform(test_df[col])

    for df in (train_df, valid_df, test_df):
        df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce").fillna(0)
        df["timestamp_days"] = (df["timestamp"] / 86400).astype(np.float32)

    feature_cols = [
        "user_id_idx", "item_id_idx", "price", "sales_rank",
        "brand_idx", "primary_category_idx", "timestamp_days",
    ]

    # Build item lookup for scoring new data
    item_lookup = item_df[["item_id", "price", "sales_rank", "brand", "primary_category"]].copy()
    
    # Handle unknown brands/categories by replacing with 'Unknown'
    known_brands = set(encoders["brand"].classes_)
    known_cats = set(encoders["primary_category"].classes_)
    
    item_lookup["brand"] = item_lookup["brand"].astype(str).apply(
        lambda x: x if x in known_brands else "Unknown"
    )
    item_lookup["primary_category"] = item_lookup["primary_category"].astype(str).apply(
        lambda x: x if x in known_cats else "Unknown"
    )
    
    item_lookup["brand_idx"] = encoders["brand"].transform(item_lookup["brand"])
    item_lookup["primary_category_idx"] = encoders["primary_category"].transform(item_lookup["primary_category"])

    return train_df, valid_df, test_df, feature_cols, encoders, item_lookup


def run_pipeline():
    print("Loading interactions + metadata")
    train_df = load_interactions("train")
    valid_df = load_interactions("valid")
    test_df = load_interactions("test")
    item_df = load_item_features()

    print(f"Sampling {NUM_NEG}:1 negatives per positive")
    all_items = item_df["item_id"].unique().tolist()
    user_pos_items = build_user_pos_items([train_df, valid_df, test_df])
    train_df_ns = sample_negatives(train_df, user_pos_items, all_items, NUM_NEG, SEED)
    valid_df_ns = sample_negatives(valid_df, user_pos_items, all_items, NUM_NEG, VALID_SEED)
    test_df_ns = sample_negatives(test_df, user_pos_items, all_items, NUM_NEG, TEST_SEED)

    print("Encoding categorical + numerical features")
    train_df_enc, valid_df_enc, test_df_enc, feature_cols, encoders, item_lookup = encode_features_v2(
        train_df_ns, valid_df_ns, test_df_ns, item_df
    )

    print("Training XGBoost")
    model = train_xgboost(train_df_enc, valid_df_enc, feature_cols)

    print("Evaluating")
    train_metrics = evaluate_model(model, train_df_enc, feature_cols)
    valid_metrics = evaluate_model(model, valid_df_enc, feature_cols)
    test_metrics = evaluate_model(model, test_df_enc, feature_cols)

    results = {
        "train": train_metrics,
        "valid": valid_metrics,
        "test": test_metrics,
        "feature_cols": feature_cols,
        "params": {
            "num_neg": NUM_NEG,
            "seed": SEED,
            "tree_method": model.attributes().get("tree_method", "unknown"),
        },
    }

    model_path = os.path.join(OUTPUT_DIR, "ranking_xgboost.model")
    metrics_path = os.path.join(OUTPUT_DIR, "ranking_xgboost_results.json")

    model.save_model(model_path)
    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)

    print(json.dumps(results, indent=2))
    print(f"\nModel saved to: {model_path}")
    print(f"Metrics saved to: {metrics_path}")
    
    return {
        "results": results,
        "model": model,
        "encoders": encoders,
        "item_lookup": item_lookup,
        "feature_cols": feature_cols,
    }



In [7]:
%%time
artifacts = run_pipeline()
xgb_results = artifacts["results"]
xgb_model = artifacts["model"]
xgb_encoders = artifacts["encoders"]
xgb_item_lookup = artifacts["item_lookup"]
xgb_feature_cols = artifacts["feature_cols"]



Loading interactions + metadata
Sampling 1:1 negatives per positive
Encoding categorical + numerical features
Training XGBoost
GPU detected - using device=cuda:0
[0]	train-auc:0.88529	train-logloss:0.66986	valid-auc:0.66900	valid-logloss:0.68549
[50]	train-auc:0.91842	train-logloss:0.41477	valid-auc:0.70498	valid-logloss:0.63724
[72]	train-auc:0.92275	train-logloss:0.38859	valid-auc:0.70828	valid-logloss:0.63922
Evaluating


/tmp/ipykernel_3495347/3176133740.py:103: UserWarning: [00:13:23] WARNING: /workspace/src/c_api/c_api.cc:1575: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.
  model.save_model(model_path)


{
  "train": {
    "AUC": 0.9227484628508229,
    "LogLoss": 0.38859069988431794
  },
  "valid": {
    "AUC": 0.708275742066521,
    "LogLoss": 0.6392213138151955
  },
  "test": {
    "AUC": 0.7025020673543936,
    "LogLoss": 0.6557430509004429
  },
  "feature_cols": [
    "user_id_idx",
    "item_id_idx",
    "price",
    "sales_rank",
    "brand_idx",
    "primary_category_idx",
    "timestamp_days"
  ],
  "params": {
    "num_neg": 1,
    "seed": 42,
    "tree_method": "unknown"
  }
}

Model saved to: saved_models/ranking_xgboost.model
Metrics saved to: saved_models/ranking_xgboost_results.json
CPU times: user 37.7 s, sys: 1.42 s, total: 39.1 s
Wall time: 39.1 s


 Train AUC 0.92 → Valid/Test AUC 0.70: The model learns collaborative filtering patterns (which users buy which items), but overfits somewhat to training data

Test AUC 0.70: The model correctly ranks the positive item above a random negative 70% of the time, this seems to be a reasonable result for a recommendation task

Early stopping triggered at round 72 (out of 500), indicating the model started overfitting

In [8]:
# === SAVE trained XGBoost artifacts for later reuse ===
import pickle

SAVE_DIR = os.path.join(OUTPUT_DIR, "xgboost")
os.makedirs(SAVE_DIR, exist_ok=True)

# Save encoders (LabelEncoders for user_id, item_id, brand, category)
with open(os.path.join(SAVE_DIR, "xgb_encoders.pkl"), "wb") as f:
    pickle.dump(xgb_encoders, f)

# Save item lookup table (pre-encoded item features)
xgb_item_lookup.to_pickle(os.path.join(SAVE_DIR, "xgb_item_lookup.pkl"))

# Save feature columns list
with open(os.path.join(SAVE_DIR, "xgb_feature_cols.pkl"), "wb") as f:
    pickle.dump(xgb_feature_cols, f)

# Model is already saved by run_pipeline(); copy to our save dir
import shutil
src_model = os.path.join(OUTPUT_DIR, "ranking_xgboost.model")
dst_model = os.path.join(SAVE_DIR, "ranking_xgboost.model")
if os.path.exists(src_model):
    shutil.copy(src_model, dst_model)

print(f"Saved XGBoost artifacts to {SAVE_DIR}/")
print(f"  - xgb_encoders.pkl")
print(f"  - xgb_item_lookup.pkl")
print(f"  - xgb_feature_cols.pkl")
print(f"  - ranking_xgboost.model")


Saved XGBoost artifacts to saved_models/xgboost/
  - xgb_encoders.pkl
  - xgb_item_lookup.pkl
  - xgb_feature_cols.pkl
  - ranking_xgboost.model


### (Optional) Load saved XGBoost artifacts — skip retraining

Run this cell **instead of** the training cell if already have `saved_models/xgboost/` from a previous run.


In [ ]:
# === LOAD saved XGBoost artifacts (skip retraining) ===
# Uncomment and run this cell to skip training and use a saved model.

# import pickle

# SAVE_DIR = os.path.join(OUTPUT_DIR, "xgboost")

# # Load encoders
# with open(os.path.join(SAVE_DIR, "xgb_encoders.pkl"), "rb") as f:
#     xgb_encoders = pickle.load(f)

# # Load item lookup
# xgb_item_lookup = pd.read_pickle(os.path.join(SAVE_DIR, "xgb_item_lookup.pkl"))

# # Load feature columns
# with open(os.path.join(SAVE_DIR, "xgb_feature_cols.pkl"), "rb") as f:
#     xgb_feature_cols = pickle.load(f)

# # Load model
# xgb_model = xgb.Booster()
# xgb_model.load_model(os.path.join(SAVE_DIR, "ranking_xgboost.model"))

# print(f"Loaded XGBoost from {SAVE_DIR}/")
# print(f"  Encoders: {list(xgb_encoders.keys())}")
# print(f"  Item lookup: {len(xgb_item_lookup)} items")
# print(f"  Features: {xgb_feature_cols}")


## Experiment: Remove user_id_idx and item_id_idx

 The previous run showed **AUC = 1.0** on all splits, which indicates data leakage. The model memorized (user, item) pairs instead of learning generalizable patterns.

 **Hypothesis**: Removing `user_id_idx` and `item_id_idx` will force the model to learn from content features only (price, sales_rank, brand, category, timestamp), resulting in realistic performance.

In [9]:
# Experiment: Content-only features (no user/item IDs)
def run_experiment_no_ids():
    print("=" * 60)
    print("EXPERIMENT: XGBoost with content features only (no IDs)")
    print("=" * 60)
    
    print("\nLoading interactions + metadata")
    train_df = load_interactions("train")
    valid_df = load_interactions("valid")
    test_df = load_interactions("test")
    item_df = load_item_features()

    print(f"Sampling {NUM_NEG}:1 negatives per positive")
    all_items = item_df["item_id"].unique().tolist()
    user_pos_items = build_user_pos_items([train_df, valid_df, test_df])
    train_df_ns = sample_negatives(train_df, user_pos_items, all_items, NUM_NEG, SEED)
    valid_df_ns = sample_negatives(valid_df, user_pos_items, all_items, NUM_NEG, VALID_SEED)
    test_df_ns = sample_negatives(test_df, user_pos_items, all_items, NUM_NEG, TEST_SEED)

    print("Encoding features")
    train_df_enc, valid_df_enc, test_df_enc, _ = encode_features(
        train_df_ns, valid_df_ns, test_df_ns, item_df
    )

    # Content-only features (REMOVED user_id_idx and item_id_idx)
    feature_cols_no_ids = [
        "price",
        "sales_rank",
        "brand_idx",
        "primary_category_idx",
        "timestamp_days",
    ]
    
    print(f"\nFeatures used: {feature_cols_no_ids}")
    print(f"Number of features: {len(feature_cols_no_ids)}")

    print("\nTraining XGBoost (content features only)")
    model = train_xgboost(train_df_enc, valid_df_enc, feature_cols_no_ids)

    print("\nEvaluating")
    train_metrics = evaluate_model(model, train_df_enc, feature_cols_no_ids)
    valid_metrics = evaluate_model(model, valid_df_enc, feature_cols_no_ids)
    test_metrics = evaluate_model(model, test_df_enc, feature_cols_no_ids)

    results = {
        "experiment": "content_features_only",
        "train": train_metrics,
        "valid": valid_metrics,
        "test": test_metrics,
        "feature_cols": feature_cols_no_ids,
    }

    print("\n" + "=" * 60)
    print("RESULTS (Content Features Only)")
    print("=" * 60)
    print(json.dumps(results, indent=2))
    
    # Save experiment results
    exp_path = os.path.join(OUTPUT_DIR, "ranking_xgboost_no_ids_results.json")
    with open(exp_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)
    print(f"\nSaved to: {exp_path}")
    
    return results


In [10]:
%%time
exp_results = run_experiment_no_ids()


EXPERIMENT: XGBoost with content features only (no IDs)

Loading interactions + metadata
Sampling 1:1 negatives per positive
Encoding features

Features used: ['price', 'sales_rank', 'brand_idx', 'primary_category_idx', 'timestamp_days']
Number of features: 5

Training XGBoost (content features only)
GPU detected - using device=cuda:0
[0]	train-auc:0.57047	train-logloss:0.69214	valid-auc:0.56284	valid-logloss:0.69239
[50]	train-auc:0.61499	train-logloss:0.67458	valid-auc:0.60171	valid-logloss:0.67854
[100]	train-auc:0.63780	train-logloss:0.66546	valid-auc:0.62047	valid-logloss:0.67131
[150]	train-auc:0.65473	train-logloss:0.65865	valid-auc:0.63437	valid-logloss:0.66598
[200]	train-auc:0.66776	train-logloss:0.65274	valid-auc:0.64505	valid-logloss:0.66130
[250]	train-auc:0.67708	train-logloss:0.64790	valid-auc:0.65273	valid-logloss:0.65747
[300]	train-auc:0.68531	train-logloss:0.64348	valid-auc:0.65950	valid-logloss:0.65400
[350]	train-auc:0.69225	train-logloss:0.63953	valid-auc:0.66522	

This is the complete results from last cell: 
============================================================
EXPERIMENT: XGBoost with content features only (no IDs)
============================================================

Loading interactions + metadata
Warning: 1 users could not be mapped in train
Warning: 1 items could not be mapped in valid
Sampling 1:1 negatives per positive
Encoding features

Features used: ['price', 'sales_rank', 'brand_idx', 'primary_category_idx', 'timestamp_days']
Number of features: 5

Training XGBoost (content features only)
GPU detected - using device=cuda:0
[0]	train-auc:0.57047	train-logloss:0.69214	valid-auc:0.56284	valid-logloss:0.69239
[50]	train-auc:0.61499	train-logloss:0.67458	valid-auc:0.60171	valid-logloss:0.67854
[100]	train-auc:0.63780	train-logloss:0.66546	valid-auc:0.62047	valid-logloss:0.67131
[150]	train-auc:0.65473	train-logloss:0.65865	valid-auc:0.63437	valid-logloss:0.66598
[200]	train-auc:0.66776	train-logloss:0.65274	valid-auc:0.64505	valid-logloss:0.66130
[250]	train-auc:0.67708	train-logloss:0.64790	valid-auc:0.65273	valid-logloss:0.65747
[300]	train-auc:0.68531	train-logloss:0.64348	valid-auc:0.65950	valid-logloss:0.65400
[350]	train-auc:0.69225	train-logloss:0.63953	valid-auc:0.66522	valid-logloss:0.65088
[400]	train-auc:0.69816	train-logloss:0.63602	valid-auc:0.66995	valid-logloss:0.64815
[450]	train-auc:0.70388	train-logloss:0.63258	valid-auc:0.67480	valid-logloss:0.64545
[499]	train-auc:0.70910	train-logloss:0.62935	valid-auc:0.67916	valid-logloss:0.64296

Evaluating

============================================================
RESULTS (Content Features Only)
============================================================
{
  "experiment": "content_features_only",
  "train": {
    "AUC": 0.7090985083779296,
    "LogLoss": 0.6293457280324073
  },
  "valid": {
    "AUC": 0.6791596711162855,
    "LogLoss": 0.6429595428609834
  },
  "test": {
    "AUC": 0.6635762791418683,
    "LogLoss": 0.6488522228288706
  },
  "feature_cols": [
    "price",
    "sales_rank",
    "brand_idx",
    "primary_category_idx",
    "timestamp_days"
  ]
}

Saved to: saved_models/ranking_xgboost_no_ids_results.json
CPU times: user 46.2 s, sys: 1.01 s, total: 47.2 s
Wall time: 47.2 s

Content-Only XGBoost Results:

Minimal Overfitting 

Train AUC (0.71) ≈ Valid AUC (0.68) ≈ Test AUC (0.66)

The small gap (~3-4%) indicates the model generalizes well.
No early stopping triggered, ran all 500 rounds, meaning valid loss kept improving.

2. Content Features Have Predictive Value 
AUC 0.66 > 0.50 (random): The model is better than random guessing.
Using only 5 features (price, sales_rank, brand, category, timestamp), the model correctly ranks positives above negatives 66% of the time.

3. Without knowing who the user is or which specific item it is, the model can still predict:

Items with certain price ranges are more likely to be purchased

Items with better sales_rank (more popular) tend to be preferred

Certain brands and categories have higher conversion rates

User/item IDs add 4% AUC improvement over content features alone. This means:

66% of the signal comes from item properties (price, popularity, brand)

4% comes from personalization (which users prefer which items)

In [ ]:
## Experiment 2: Negatives from Items with Interactions Only

# The previous experiment still showed AUC = 1.0 because:
# - **Positive items** = Items someone bought (have good sales_rank, popular brands)
# - **Random negatives** = Many "dead" items nobody ever bought

# **Fix**: Sample negatives only from items that have at least one interaction. 
# This creates a more realistic task: Given two items that people actually buy, which one will this user prefer?


In [11]:
# Experiment 2: Sample negatives only from items with interactions
def sample_negatives_from_interacted(df, user_pos_items, interacted_items, num_neg=1, seed=42):
    """Sample negatives only from items that have at least one interaction.
    
    This creates realistic negatives: instead of random catalog items (many "dead"),
    we sample from items that OTHER users have bought.
    """
    rng = random.Random(seed)
    interacted_list = list(interacted_items)
    negatives = []
    
    for row in df.itertuples(index=False):
        user = row.user_id
        for _ in range(num_neg):
            attempts = 0
            while attempts < 100:  # Prevent infinite loop
                neg_item = rng.choice(interacted_list)
                if neg_item not in user_pos_items[user]:
                    negatives.append({
                        "user_id": user,
                        "item_id": neg_item,
                        "timestamp": row.timestamp,
                        "label": 0,
                    })
                    break
                attempts += 1
    
    neg_df = pd.DataFrame(negatives)
    return pd.concat([df, neg_df], ignore_index=True)


def run_experiment_hard_negatives():
    print("=" * 70)
    print("EXPERIMENT 2: Negatives from items with interactions only")
    print("=" * 70)
    
    print("\nLoading interactions + metadata")
    train_df = load_interactions("train")
    valid_df = load_interactions("valid")
    test_df = load_interactions("test")
    item_df = load_item_features()

    # Get items that have at least one interaction (across all splits)
    all_interacted_items = set(train_df["item_id"].unique()) | \
                           set(valid_df["item_id"].unique()) | \
                           set(test_df["item_id"].unique())
    
    all_catalog_items = set(item_df["item_id"].unique())
    
    print(f"\nItem statistics:")
    print(f"  Total items in catalog: {len(all_catalog_items):,}")
    print(f"  Items with ≥1 interaction: {len(all_interacted_items):,}")
    print(f"  Items with 0 interactions: {len(all_catalog_items - all_interacted_items):,}")
    print(f"  Negatives will be sampled from {len(all_interacted_items):,} 'real' items")

    user_pos_items = build_user_pos_items([train_df, valid_df, test_df])
    
    print(f"\nSampling {NUM_NEG}:1 hard negatives (from interacted items only)")
    train_df_ns = sample_negatives_from_interacted(train_df, user_pos_items, all_interacted_items, NUM_NEG, SEED)
    valid_df_ns = sample_negatives_from_interacted(valid_df, user_pos_items, all_interacted_items, NUM_NEG, VALID_SEED)
    test_df_ns = sample_negatives_from_interacted(test_df, user_pos_items, all_interacted_items, NUM_NEG, TEST_SEED)

    # convert item_id to string in all dataframes before merge
    for df in (train_df_ns, valid_df_ns, test_df_ns):
        df["item_id"] = df["item_id"].astype(str)
    item_df["item_id"] = item_df["item_id"].astype(str)

    print("Encoding features")
    train_df_enc, valid_df_enc, test_df_enc, _ = encode_features(
        train_df_ns, valid_df_ns, test_df_ns, item_df
    )

    # Test both feature sets
    feature_cols_with_ids = [
        "user_id_idx", "item_id_idx", "price", "sales_rank",
        "brand_idx", "primary_category_idx", "timestamp_days",
    ]
    feature_cols_no_ids = [
        "price", "sales_rank", "brand_idx", "primary_category_idx", "timestamp_days",
    ]

    results = {}
    
    # --- With IDs ---
    print("\n" + "-" * 50)
    print("Training XGBoost WITH user/item IDs")
    print("-" * 50)
    model_ids = train_xgboost(train_df_enc, valid_df_enc, feature_cols_with_ids)
    results["with_ids"] = {
        "train": evaluate_model(model_ids, train_df_enc, feature_cols_with_ids),
        "valid": evaluate_model(model_ids, valid_df_enc, feature_cols_with_ids),
        "test": evaluate_model(model_ids, test_df_enc, feature_cols_with_ids),
    }
    
    # --- Without IDs ---
    print("\n" + "-" * 50)
    print("Training XGBoost WITHOUT user/item IDs (content only)")
    print("-" * 50)
    model_no_ids = train_xgboost(train_df_enc, valid_df_enc, feature_cols_no_ids)
    results["without_ids"] = {
        "train": evaluate_model(model_no_ids, train_df_enc, feature_cols_no_ids),
        "valid": evaluate_model(model_no_ids, valid_df_enc, feature_cols_no_ids),
        "test": evaluate_model(model_no_ids, test_df_enc, feature_cols_no_ids),
    }

    print("\n" + "=" * 70)
    print("RESULTS COMPARISON (Hard Negatives)")
    print("=" * 70)
    print("\n WITH user/item IDs:")
    print(f"   Train AUC: {results['with_ids']['train']['AUC']:.4f}")
    print(f"   Valid AUC: {results['with_ids']['valid']['AUC']:.4f}")
    print(f"   Test AUC:  {results['with_ids']['test']['AUC']:.4f}")
    
    print("\n WITHOUT user/item IDs (content only):")
    print(f"   Train AUC: {results['without_ids']['train']['AUC']:.4f}")
    print(f"   Valid AUC: {results['without_ids']['valid']['AUC']:.4f}")
    print(f"   Test AUC:  {results['without_ids']['test']['AUC']:.4f}")
    
    # Save results
    exp_path = os.path.join(OUTPUT_DIR, "ranking_xgboost_hard_negatives.json")
    with open(exp_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)
    print(f"\nSaved to: {exp_path}")
    
    return results


In [12]:
%%time
hard_neg_results = run_experiment_hard_negatives()


EXPERIMENT 2: Negatives from items with interactions only

Loading interactions + metadata

Item statistics:
  Total items in catalog: 259,204
  Items with ≥1 interaction: 249,273
  Items with 0 interactions: 9,931
  Negatives will be sampled from 249,273 'real' items

Sampling 1:1 hard negatives (from interacted items only)
Encoding features

--------------------------------------------------
Training XGBoost WITH user/item IDs
--------------------------------------------------
GPU detected - using device=cuda:0
[0]	train-auc:0.87801	train-logloss:0.67100	valid-auc:0.66210	valid-logloss:0.68588
[50]	train-auc:0.91290	train-logloss:0.42703	valid-auc:0.69410	valid-logloss:0.63675
[89]	train-auc:0.92401	train-logloss:0.37276	valid-auc:0.70806	valid-logloss:0.63923

--------------------------------------------------
Training XGBoost WITHOUT user/item IDs (content only)
--------------------------------------------------
GPU detected - using device=cuda:0
[0]	train-auc:0.56813	train-logloss

This is the full results from last cell: 
======================================================================
EXPERIMENT 2: Negatives from items with interactions only
======================================================================

Loading interactions + metadata
Warning: 1 users could not be mapped in train
Warning: 1 items could not be mapped in valid

Item statistics:
  Total items in catalog: 259,204
  Items with ≥1 interaction: 249,273
  Items with 0 interactions: 9,931
  Negatives will be sampled from 249,273 'real' items

Sampling 1:1 hard negatives (from interacted items only)
Encoding features

--------------------------------------------------
Training XGBoost WITH user/item IDs
--------------------------------------------------
GPU detected - using device=cuda:0
[0]	train-auc:0.87801	train-logloss:0.67100	valid-auc:0.66210	valid-logloss:0.68588
[50]	train-auc:0.91290	train-logloss:0.42703	valid-auc:0.69410	valid-logloss:0.63675
[89]	train-auc:0.92401	train-logloss:0.37276	valid-auc:0.70806	valid-logloss:0.63923

--------------------------------------------------
Training XGBoost WITHOUT user/item IDs (content only)
--------------------------------------------------
GPU detected - using device=cuda:0
[0]	train-auc:0.56813	train-logloss:0.69220	valid-auc:0.56187	valid-logloss:0.69240
[50]	train-auc:0.61629	train-logloss:0.67415	valid-auc:0.60205	valid-logloss:0.67833
[100]	train-auc:0.63899	train-logloss:0.66516	valid-auc:0.62139	valid-logloss:0.67105
[150]	train-auc:0.65517	train-logloss:0.65836	valid-auc:0.63458	valid-logloss:0.66570
[200]	train-auc:0.66769	train-logloss:0.65280	valid-auc:0.64519	valid-logloss:0.66123
[250]	train-auc:0.67740	train-logloss:0.64780	valid-auc:0.65320	valid-logloss:0.65733
[300]	train-auc:0.68522	train-logloss:0.64348	valid-auc:0.65954	valid-logloss:0.65394
[350]	train-auc:0.69180	train-logloss:0.63952	valid-auc:0.66510	valid-logloss:0.65083
[400]	train-auc:0.69719	train-logloss:0.63617	valid-auc:0.66958	valid-logloss:0.64817
[450]	train-auc:0.70307	train-logloss:0.63263	valid-auc:0.67414	valid-logloss:0.64543
[499]	train-auc:0.70807	train-logloss:0.62943	valid-auc:0.67825	valid-logloss:0.64294

======================================================================
RESULTS COMPARISON (Hard Negatives)
======================================================================

 WITH user/item IDs:
   Train AUC: 0.9240
   Valid AUC: 0.7081
   Test AUC:  0.7006

 WITHOUT user/item IDs (content only):
   Train AUC: 0.7081
   Valid AUC: 0.6782
   Test AUC:  0.6628

Saved to: saved_models/ranking_xgboost_hard_negatives.json
CPU times: user 51 s, sys: 1.32 s, total: 52.4 s
Wall time: 52.2 s

Hard Negatives Experiment:

Instead of sampling negatives from all catalog items (259K), this experiment samples negatives only from items that have at least one interaction (249K). This creates "harder" negatives because:

Random catalog negatives include many "dead" items nobody ever buys.
Hard negatives are items that other users actually purchased, making them more realistic competitors.

1. With User/Item IDs

Train AUC 0.92 → Test AUC 0.70: Some overfitting, but model still generalizes

Early stopping at round 89 (of 500) - model learned quickly then started overfitting

Test AUC 0.70: Correctly ranks the user's true purchase above a "hard" negative 70% of the time

2. Content-Only (No IDs)
Train ≈ Valid ≈ Test (~0.66-0.71): Minimal overfitting

Ran all 500 rounds - content features provide steady but limited signal

Test AUC 0.66: Without personalization, content features alone rank correctly 66% of the time

3. Hard vs Random Negatives
Comparing to the main experiment (random negatives):

Results are nearly identical. This means:

The 10K "dead" items in the catalog don't significantly affect the evaluation

The model's performance is consistent regardless of negative sampling strategy


Hard Negatives Experiment

Similar results to the main experiment, sampling negatives from "real" items (those with interactions) vs. random catalog items didn't significantly change outcomes.



## Shared random‑negative evaluation (XGBoost)

This cell evaluates XGBoost under the **same random‑negative protocol** used by the other models (Most Popular, Item‑KNN, Two‑Tower, DeepFM):

- It loads the original RecBole `.inter` splits (`train`, `test`).
- It calls `build_random_neg_eval_df` from `eval_utils.py` to build a shared eval set with **1 positive + 5 random negatives per user**.
- It scores each `(user, item)` pair with XGBoost and computes:
  - `recall@10`, `recall@20`
  - `ndcg@10`, `ndcg@20`
  - `num_users_eval`
- These metrics are the ones to use when **comparing XGBoost against other models**, because they all run on this same eval set.


In [ ]:
from eval_utils import build_random_neg_eval_df, evaluate_recall_ndcg_at_k
import sys

# === CONFIGURABLE PARAMETERS  ===
NUM_NEG = 5           # negatives per user
MAX_USERS = 10000     # number of users to evaluate  
SEED = 42             # random seed
# ======================================================================

print(f"Building eval_df: {MAX_USERS} users, {NUM_NEG} negatives, seed={SEED}")
sys.stdout.flush()

# 1) Build eval_df from TRAINING DATA
# load_interactions() now converts RecBole integer IDs to original tokens (ASINs)
print("  Loading training data")
sys.stdout.flush()
train_df_eval = load_interactions("train")
test_df_eval = load_interactions("test")

train_df_eval["label"] = pd.to_numeric(train_df_eval["label"], errors="coerce").fillna(0).astype(int)
test_df_eval["label"] = pd.to_numeric(test_df_eval["label"], errors="coerce").fillna(0).astype(int)

print(f"  Train positives: {(train_df_eval['label']==1).sum()}")
print(f"  Test positives: {(test_df_eval['label']==1).sum()}")
print(f"  Sample item_ids (should be ASINs): {train_df_eval['item_id'].head(3).tolist()}")
sys.stdout.flush()

# Use eval_utils helper to build consistent random-negative set
print("  Building eval_df with random negatives")
sys.stdout.flush()
eval_df = build_random_neg_eval_df(train_df_eval, test_df_eval, num_neg=NUM_NEG, seed=SEED, max_users=MAX_USERS)
print(f"Built eval_df: {len(eval_df)} rows, {eval_df['user_id'].nunique()} users")
print(f"Sample user_ids: {eval_df['user_id'].head(3).tolist()}")
print(f"Sample item_ids (ASINs): {eval_df['item_id'].head(3).tolist()}")
sys.stdout.flush()

# Check encoder classes to understand what format they expect
print(f"\\nEncoder user_id classes sample: {list(xgb_encoders['user_id'].classes_[:5])}")
print(f"Encoder item_id classes sample: {list(xgb_encoders['item_id'].classes_[:5])}")
sys.stdout.flush()


# 2) Score with XGBoost (vectorized encoding)
def score_with_xgboost(eval_df, model, encoders, item_lookup, feature_cols):
    print("  Encoding features (vectorized)")
    sys.stdout.flush()
    
    df = eval_df.copy()
    # Convert to strings for encoder lookup
    df["user_id"] = df["user_id"].astype(str)
    df["item_id"] = df["item_id"].astype(str)
    
    print("    Building user_id index")
    sys.stdout.flush()
    user_to_idx = {str(u): i for i, u in enumerate(encoders["user_id"].classes_)}
    df["user_id_idx"] = df["user_id"].map(user_to_idx).fillna(-1).astype(int)
    user_matched = (df["user_id_idx"] >= 0).sum()
    print(f"    Users matched: {user_matched} / {len(df)}")
    
    print("    Merging item features")
    sys.stdout.flush()
    # item_lookup has ASIN as item_id (from training pipeline)
    df = df.merge(item_lookup, on="item_id", how="left")

    for col, default in [
        ("price", 0.0),
        ("sales_rank", 0.0),
        ("brand_idx", 0),
        ("primary_category_idx", 0),
    ]:
        if col not in df:
            df[col] = default
        else:
            df[col] = df[col].fillna(default)
    
    print("    Building item_id index")
    sys.stdout.flush()
    item_to_idx = {str(i): idx for idx, i in enumerate(encoders["item_id"].classes_)}
    df["item_id_idx"] = df["item_id"].map(item_to_idx).fillna(-1).astype(int)
    item_matched = (df["item_id_idx"] >= 0).sum()
    price_valid = df["price"].notna().sum()
    print(f"    Items matched: {item_matched} / {len(df)}")
    print(f"    Prices valid: {price_valid} / {len(df)}")
    sys.stdout.flush()
    
    df["timestamp_days"] = 0.0
    
    mask = (df["user_id_idx"] >= 0) & (df["item_id_idx"] >= 0)
    df_valid = df[mask].copy()
    print(f"  Valid rows after encoding: {len(df_valid)} / {len(df)}")
    sys.stdout.flush()
    
    if df_valid.empty:
        raise RuntimeError("No valid rows for XGBoost scoring.")
    
    # --- Diagnostics: feature ranges ---
    print("  Feature ranges (df_valid)")
    for col in feature_cols:
        col_min = df_valid[col].min()
        col_max = df_valid[col].max()
        nunique = df_valid[col].nunique()
        print(f"    {col}: min={col_min}, max={col_max}, nunique={nunique}, dtype={df_valid[col].dtype}")
    sys.stdout.flush()
    
    # Score with XGBoost
    print("  Scoring with XGBoost")
    sys.stdout.flush()
    dmatrix = xgb.DMatrix(df_valid[feature_cols])
    scores = model.predict(dmatrix)
    df_valid["score_xgb"] = scores
    print("  Done")
    sys.stdout.flush()
    
    return df_valid

print("\\nScoring with XGBoost")
sys.stdout.flush()
eval_xgb = score_with_xgboost(eval_df, xgb_model, xgb_encoders, xgb_item_lookup, xgb_feature_cols)

# --- Diagnostics: score distribution and margins ---
print("\\nDEBUG: score_xgb summary by label")
print(eval_xgb.groupby("label")["score_xgb"].describe())

pos_scores = eval_xgb[eval_xgb["label"] == 1].set_index("user_id")["score_xgb"]
neg_max = eval_xgb[eval_xgb["label"] == 0].groupby("user_id")["score_xgb"].max()
common_users = pos_scores.index.intersection(neg_max.index)
pos_scores = pos_scores.loc[common_users]
neg_max = neg_max.loc[common_users]
beats = (pos_scores > neg_max).mean()
beats_ge = (pos_scores >= neg_max).mean()
worst_margins = (pos_scores - neg_max).nsmallest(5)
print(f"Fraction pos > max_neg: {beats:.4f}; pos >= max_neg: {beats_ge:.4f}")
print("Smallest margins (pos - max_neg):", worst_margins.tolist())

print("\\nComputing metrics")
sys.stdout.flush()
xgb_metrics = evaluate_recall_ndcg_at_k(eval_xgb, "score_xgb", ks=(10, 20))

def evaluate_recall_ndcg_tierobust(df, score_col, ks=(10, 20)):
    metrics = {f"recall@{k}": 0.0 for k in ks}
    metrics.update({f"ndcg@{k}": 0.0 for k in ks})
    users = 0
    for _, g in df.groupby("user_id"):
        users += 1
        g_sorted = g.sort_values(score_col, ascending=False, kind="mergesort")
        g_sorted["rank"] = np.arange(1, len(g_sorted) + 1)
        pos = g_sorted[g_sorted["label"] == 1]
        if pos.empty:
            continue
        for k in ks:
            hits = (pos["rank"] <= k).sum()
            metrics[f"recall@{k}"] += hits
            metrics[f"ndcg@{k}"] += (1.0 / np.log2(pos["rank"] + 1)).sum()
    if users == 0:
        metrics["num_users"] = 0.0
        return metrics
    for k in ks:
        metrics[f"recall@{k}"] /= users
        metrics[f"ndcg@{k}"] /= users
    metrics["num_users"] = float(users)
    return metrics

xgb_metrics_tierobust = evaluate_recall_ndcg_tierobust(eval_xgb, "score_xgb", ks=(10, 20))

print("\\n=== XGBoost (random-neg eval) ===")
for k, v in xgb_metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

print("\\n=== XGBoost (tie-robust) ===")
for k, v in xgb_metrics_tierobust.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

Building eval_df: 10000 users, 5 negatives, seed=42
  Loading training data...
  Train positives: 1541320
  Test positives: 328812
  Sample item_ids (should be ASINs): ['B000005L8T', 'B0000065GY', 'B00000IR6P']
  Building eval_df with random negatives...
  Filtering positives
  Item universe: 239,937 items
  Getting unique test users
  Evaluating 10,000 users with 5 negatives each
  Building seen-items index
  Sampling negatives
    Progress: 0/10,000 (0%)
    Progress: 1,000/10,000 (10%)
    Progress: 2,000/10,000 (20%)
    Progress: 3,000/10,000 (30%)
    Progress: 4,000/10,000 (40%)
    Progress: 5,000/10,000 (50%)
    Progress: 6,000/10,000 (60%)
    Progress: 7,000/10,000 (70%)
    Progress: 8,000/10,000 (80%)
    Progress: 9,000/10,000 (90%)
    Progress: 10,000/10,000 (100%)
  Built eval_df: 60,000 rows
Built eval_df: 60000 rows, 10000 users
Sample user_ids: ['A2TWLGRG77HAH1', 'A2TWLGRG77HAH1', 'A2TWLGRG77HAH1']
Sample item_ids (ASINs): ['B001U9O3N0', 'B000IZA732', 'B000K74RRA']

Here is the full results of the last cell:
Building eval_df: 10000 users, 5 negatives, seed=42
  Loading training data...
Warning: 1 users could not be mapped in train
  Train positives: 1541320
  Test positives: 328812
  Sample item_ids (should be ASINs): ['B000005L8T', 'B0000065GY', 'B00000IR6P']
  Building eval_df with random negatives...
  Filtering positives
  Item universe: 239,937 items
  Getting unique test users
  Evaluating 10,000 users with 5 negatives each
  Building seen-items index
  Sampling negatives
    Progress: 0/10,000 (0%)
    Progress: 1,000/10,000 (10%)
    Progress: 2,000/10,000 (20%)
    Progress: 3,000/10,000 (30%)
    Progress: 4,000/10,000 (40%)
    Progress: 5,000/10,000 (50%)
    Progress: 6,000/10,000 (60%)
    Progress: 7,000/10,000 (70%)
    Progress: 8,000/10,000 (80%)
    Progress: 9,000/10,000 (90%)
    Progress: 10,000/10,000 (100%)
  Built eval_df: 60,000 rows
Built eval_df: 60000 rows, 10000 users
Sample user_ids: ['A2TWLGRG77HAH1', 'A2TWLGRG77HAH1', 'A2TWLGRG77HAH1']
Sample item_ids (ASINs): ['B001U9O3N0', 'B000IZA732', 'B000K74RRA']
\nEncoder user_id classes sample: ['A000186437REL8X2RW8UW', 'A0002574WYJMBWKNCPY8', 'A00029263J863WSR0TDRS', 'A00031961JI1CBNV98TW', 'A000325234LCBTFVL1QK4']
Encoder item_id classes sample: ['0205616461', '0558925278', '0733001998', '0737104473', '0762451459']
\nScoring with XGBoost...
  Encoding features (vectorized)...
    Building user_id index...
    Users matched: 60000 / 60000
    Merging item features...
    Building item_id index...
    Items matched: 60000 / 60000
    Prices valid: 60000 / 60000
  Valid rows after encoding: 60000 / 60000
  Feature ranges (df_valid)
    user_id_idx: min=10, max=1209975, nunique=10000, dtype=int64
    item_id_idx: min=1, max=259191, nunique=51858, dtype=int64
    price: min=0.01, max=999.0, nunique=5727, dtype=float64
    sales_rank: min=5.0, max=5059371.0, nunique=48847, dtype=float64
    brand_idx: min=0, max=13184, nunique=5761, dtype=int64
    primary_category_idx: min=0, max=3, nunique=4, dtype=int64
    timestamp_days: min=0.0, max=0.0, nunique=1, dtype=float64
  Scoring with XGBoost...
  Done!
\nDEBUG: score_xgb summary by label
         count     mean       std       min       25%       50%       75%  \
label                                                                       
0      50000.0  0.18270  0.194767  0.029261  0.072358  0.098637  0.171762   
1      10000.0  0.30201  0.284482  0.036229  0.081321  0.138225  0.633281   

            max  
label            
0      0.941741  
1      0.950178  
Fraction pos > max_neg: 0.3222; pos >= max_neg: 0.3222
Smallest margins (pos - max_neg): [-0.8771243095397949, -0.8654510974884033, -0.8503284454345703, -0.8468666076660156, -0.8439326286315918]
\nComputing metrics...
\n=== XGBoost (random-neg eval) ===
  recall@10: 1.0000
  ndcg@10: 0.6436
  recall@20: 1.0000
  ndcg@20: 0.6436
  num_users: 10000.0000
\n=== XGBoost (tie-robust) ===
  recall@10: 1.0000
  recall@20: 1.0000
  ndcg@10: 0.6436
  ndcg@20: 0.6436
  num_users: 10000.0000

XGBoost Evaluation Results

1. Evaluation Setup

10,000 users evaluated, each with 1 positive + 5 random negatives = 6 items per user

60,000 total rows (10,000 × 6).
All users/items successfully matched to encoders (100% coverage)

Fraction pos > max_neg: 0.3222: Only 32.2% of users have their positive ranked #1 (above all 5 negatives). 

For 32% of users: positive is ranked 1st

For 68% of users: at least one negative scores higher than the positive

Why recall@10 = 1.0?

With only 6 items per user and k=10, all items fit within the top-10 cutoff:

recall@10 = "Did the positive appear in top 10?" → Always yes (only 6 items exist). This metric is saturated and not informative here.

ndcg@10 = 0.6436 

NDCG rewards positives ranked higher, on average, the positive is ranked around position 2 out of 6 items. This aligns with the diagnostic showing 32% at rank 1, with the rest distributed across ranks 2-6.
